# Parte 3b: HyDE em português com o Quati

Responsável: Kossi Sedjro Mawuli Dominique Houessou

Seminários de Computação I. Repete a baseline da SciFact (BM25 x denso x HyDE) em português, com um subconjunto do Quati (Bueno et al., 2024, STIL). O subconjunto é salvo no formato BEIR e serve também para o app (`rag_core.Config(corpus_dir="beir_data/quati_sub", idioma="pt")`).

**Como usar:** ative a GPU T4 e rode as células em ordem. O download do Quati (cerca de 1,2 GB) leva alguns minutos.

**O que é o subconjunto:** o Quati tem 1 milhão de passagens e 50 consultas com julgamentos de relevância (feitos por um LLM, em escala de 0 a 3). Usamos todas as passagens julgadas mais uma amostra aleatória de passagens não julgadas (distratores). Com menos distratores a busca fica mais fácil que no corpus completo. Por isso **os números não são comparáveis com os do artigo do Quati**. Compare só os métodos entre si.

In [ ]:
!pip -q install rank_bm25 sentence-transformers huggingface_hub
!nvidia-smi -L || echo "ATENÇÃO: sem GPU. Ative a GPU em Ambiente de execução > Alterar tipo de ambiente de execução."

GPU 0: Tesla T4 (UUID: GPU-1c3ad5f9-3781-67e4-f4dd-79e4dd84c7b2)


## 1. Configuração

In [ ]:
import os, re, json, time, random, shutil
import numpy as np
import pandas as pd

N_DISTRATORES = 20000   # passagens não julgadas sorteadas do corpus de 1 milhão (mais = mais difícil e mais lento)
TOP_K = 100
SUBCONJUNTO = "beir_data/quati_sub"

# Apenas o mE5, que é multilíngue. O Contriever é treinado para inglês.
ENCODERS = [
    {"nome": "mE5-small", "modelo": "intfloat/multilingual-e5-small",
     "q_prefix": "query: ", "d_prefix": "passage: ", "normalize": True},
]

LLM_NOME = "Qwen/Qwen2.5-1.5B-Instruct"
N_HIPOTESES = 4
MAX_NEW_TOKENS = 200
TEMPERATURA = 0.7
SEED = 42

# Dois prompts em português, para ver se o idioma da instrução importa.
# O artigo do HyDE usa, para outras línguas, uma instrução em inglês pedindo a passagem na língua alvo.
# Os dois prompts abaixo são adaptações nossas, não do artigo.
PROMPTS = {
    "PT": "Escreva um trecho de texto em português que responda à pergunta em detalhes.\nPergunta: {}\nTrecho:",
    "EN-instr": "Please write a passage in Portuguese to answer the question in detail.\nQuestion: {}\nPassage:",
}

## 2. Subconjunto do Quati em formato BEIR

Baixa os arquivos do Quati no Hugging Face (passagens, julgamentos e consultas), guarda todas as passagens julgadas e uma amostra de distratores, e grava `corpus.jsonl`, `queries.jsonl` e `qrels/test.tsv`.

In [ ]:
1'def obter_arquivos_quati():
    from huggingface_hub import hf_hub_download
    base = dict(repo_id="unicamp-dl/quati", repo_type="dataset")
    return (hf_hub_download(filename="quati_1M.tsv", **base),
            hf_hub_download(filename="qrels/quati_1M_qrels.txt", **base),
            hf_hub_download(filename="topics/quati_all_topics.tsv", **base))


def ler_qrels_quati(caminho):
    """Linhas no formato TREC: id_consulta, 0, id_passagem, nota (0 a 3)."""
    qrels = {}
    with open(caminho, encoding="utf-8") as f:
        for linha in f:
            p = linha.split()
            if len(p) < 4:
                continue
            try:
                nota = int(p[3])
            except ValueError:      # linha de cabeçalho, se houver
                continue
            qrels.setdefault(p[0], {})[p[2]] = nota
    return qrels


def ler_topicos_quati(caminho):
    topicos = {}
    with open(caminho, encoding="utf-8") as f:
        next(f)  # cabeçalho
        for linha in f:
            qid, _, texto = linha.rstrip("\n").partition("\t")
            topicos[qid.strip()] = texto.strip()
    return topicos


def amostrar_passagens(tsv, julgadas, n_distratores, seed):
    """Uma passada pelo arquivo: guarda as julgadas e sorteia n distratores (amostragem por reservatório)."""
    rng = random.Random(seed)
    guardadas, reservatorio, vistos = {}, [], 0
    with open(tsv, encoding="utf-8") as f:
        for linha in f:
            pid, _, texto = linha.rstrip("\n").partition("\t")
            if pid == "passage_id":
                continue
            if pid in julgadas:
                guardadas[pid] = texto
                continue
            vistos += 1
            if len(reservatorio) < n_distratores:
                reservatorio.append((pid, texto))
            else:
                j = rng.randrange(vistos)
                if j < n_distratores:
                    reservatorio[j] = (pid, texto)
    return guardadas, reservatorio


def construir_subconjunto(destino=SUBCONJUNTO):
    if os.path.exists(os.path.join(destino, "corpus.jsonl")):
        print("Subconjunto já existe em", destino)
        return destino
    tsv, arq_qrels, arq_topicos = obter_arquivos_quati()
    qrels = ler_qrels_quati(arq_qrels)
    topicos = ler_topicos_quati(arq_topicos)
    # só consultas com pelo menos uma passagem relevante (nota > 0) e com texto
    qrels = {q: r for q, r in qrels.items() if q in topicos and any(s > 0 for s in r.values())}
    julgadas = {p for r in qrels.values() for p in r}
    guardadas, distratores = amostrar_passagens(tsv, julgadas, N_DISTRATORES, SEED)
    os.makedirs(os.path.join(destino, "qrels"), exist_ok=True)
    with open(os.path.join(destino, "corpus.jsonl"), "w", encoding="utf-8") as f:
        for pid, texto in list(guardadas.items()) + distratores:
            f.write(json.dumps({"_id": pid, "title": "", "text": texto}, ensure_ascii=False) + "\n")
    with open(os.path.join(destino, "queries.jsonl"), "w", encoding="utf-8") as f:
        for q in qrels:
            f.write(json.dumps({"_id": q, "text": topicos[q]}, ensure_ascii=False) + "\n")
    with open(os.path.join(destino, "qrels", "test.tsv"), "w", encoding="utf-8") as f:
        f.write("query-id\tcorpus-id\tscore\n")
        for q, r in qrels.items():
            for p, s in r.items():
                if s > 0 and p in guardadas:
                    f.write(f"{q}\t{p}\t{s}\n")
    print(f"Consultas: {len(qrels)} | passagens julgadas encontradas: {len(guardadas)} de {len(julgadas)} "
          f"| distratores: {len(distratores)}")
    return destino


construir_subconjunto()

quati_1M.tsv: reconstructing file:   0%|          |  0.00B / 1.17GB            

quati_1M.tsv: downloading bytes:           |  0.00B            

quati_1M_qrels.txt:   0%|          | 0.00/69.0k [00:00<?, ?B/s]

quati_all_topics.tsv:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Consultas: 49 | passagens julgadas encontradas: 1843 de 1843 | distratores: 20000


'beir_data/quati_sub'

## 3. Carregar o subconjunto

In [ ]:
def read_beir(caminho, split="test"):
    corpus = {}
    with open(os.path.join(caminho, "corpus.jsonl"), encoding="utf-8") as f:
        for linha in f:
            d = json.loads(linha)
            corpus[d["_id"]] = {"title": d.get("title", ""), "text": d["text"]}
    queries = {}
    with open(os.path.join(caminho, "queries.jsonl"), encoding="utf-8") as f:
        for linha in f:
            d = json.loads(linha)
            queries[d["_id"]] = d["text"]
    qrels = {}
    with open(os.path.join(caminho, "qrels", f"{split}.tsv"), encoding="utf-8") as f:
        next(f)  # cabeçalho
        for linha in f:
            qid, did, score = linha.rstrip("\n").split("\t")
            if int(score) > 0:
                qrels.setdefault(qid, {})[did] = int(score)
    queries = {q: t for q, t in queries.items() if q in qrels}
    return corpus, queries, qrels


def texto_doc(d):
    return (d["title"] + " " + d["text"]).strip()

corpus, queries, qrels = read_beir(SUBCONJUNTO)
doc_ids = list(corpus)
doc_textos = [texto_doc(corpus[d]) for d in doc_ids]
qids = list(queries)

print(f"Documentos: {len(doc_ids)} | Consultas avaliadas: {len(qids)}")
print("Exemplo de consulta:", queries[qids[0]])
print("Notas de relevância:", pd.Series([s for r in qrels.values() for s in r.values()]).value_counts().sort_index().to_dict())

Documentos: 21843 | Consultas avaliadas: 49
Exemplo de consulta: Qual a maior característica da fauna brasileira?
Notas de relevância: {1: 388, 2: 318, 3: 188}


## 4. Métricas

In [ ]:
def ndcg_at_k(ranking, rels, k=10):
    dcg = sum(rels.get(d, 0) / np.log2(i + 2) for i, d in enumerate(ranking[:k]))
    ideal = sorted(rels.values(), reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def recall_at_k(ranking, rels, k=100):
    relevantes = set(rels)
    return len(relevantes & set(ranking[:k])) / len(relevantes)


def avaliar(rankings, qrels):
    qs = list(rankings)
    return {
        "nDCG@10": float(np.mean([ndcg_at_k(rankings[q], qrels[q], 10) for q in qs])),
        "Recall@100": float(np.mean([recall_at_k(rankings[q], qrels[q], 100) for q in qs])),
    }

## 5. Funções que usam modelos

In [ ]:
def load_encoder(spec):
    import torch
    from sentence_transformers import SentenceTransformer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return SentenceTransformer(spec["modelo"], device=device)


def encode_texts(modelo, textos, prefixo="", normalize=False, batch_size=64):
    entrada = [prefixo + t for t in textos]
    return modelo.encode(entrada, batch_size=batch_size, convert_to_numpy=True,
                         normalize_embeddings=normalize, show_progress_bar=False)


def load_llm(nome):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(nome)
    tok.padding_side = "left"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    modelo = AutoModelForCausalLM.from_pretrained(nome, torch_dtype=dtype).to(device)
    modelo.eval()
    return modelo, tok


def generate_hypotheses(llm, prompts, n, max_new_tokens, temperatura, batch_size=16, seed=42):
    import torch
    from tqdm.auto import tqdm
    modelo, tok = llm
    torch.manual_seed(seed)
    resultado = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="Gerando passagens"):
        lote = prompts[i:i + batch_size]
        textos = [tok.apply_chat_template([{"role": "user", "content": p}],
                                          tokenize=False, add_generation_prompt=True) for p in lote]
        entrada = tok(textos, return_tensors="pt", padding=True).to(modelo.device)
        with torch.no_grad():
            saida = modelo.generate(
                **entrada, do_sample=True, temperature=temperatura, top_p=1.0, top_k=0,
                max_new_tokens=max_new_tokens, num_return_sequences=n,
                pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
        novos = saida[:, entrada["input_ids"].shape[1]:]
        decodificados = tok.batch_decode(novos, skip_special_tokens=True)
        for j in range(len(lote)):
            resultado.append([d.strip() for d in decodificados[j * n:(j + 1) * n]])
    return resultado

## 6. Funções de busca

In [ ]:
from rank_bm25 import BM25Okapi


def tokenizar(s):
    return re.findall(r"\w+", s.lower())


def bm25_rank(bm25, qids, queries, doc_ids, k=TOP_K):
    ranking = {}
    for q in qids:
        scores = bm25.get_scores(tokenizar(queries[q]))
        topo = np.argsort(-scores)[:k]
        ranking[q] = [doc_ids[i] for i in topo]
    return ranking


def rank_from_vectors(V, doc_embs, doc_ids, qids, k=TOP_K):
    scores = V @ doc_embs.T
    topo = np.argsort(-scores, axis=1)[:, :k]
    return {q: [doc_ids[i] for i in topo[n]] for n, q in enumerate(qids)}


def hyde_vectors(modelo, spec, qids, queries, hip):
    # Regra do hyde.py: média dos vetores de [pergunta] + passagens hipotéticas.
    # A pergunta usa o prefixo de consulta; as passagens hipotéticas usam o prefixo de documento.
    n = len(hip[qids[0]])
    assert all(len(hip[q]) == n for q in qids), "Número de passagens diferente entre consultas"
    Q = encode_texts(modelo, [queries[q] for q in qids], spec["q_prefix"], spec["normalize"])
    plano = [h for q in qids for h in hip[q]]
    H = encode_texts(modelo, plano, spec["d_prefix"], spec["normalize"])
    H = H.reshape(len(qids), n, -1)
    return (Q + H.sum(axis=1)) / (n + 1)

## 7. BM25 e busca densa

In [ ]:
resultados = []
rankings = {}


def registrar(metodo, ranking, t_busca, t_geracao=None):
    m = avaliar(ranking, qrels)
    resultados.append({"Método": metodo,
                       "nDCG@10": round(m["nDCG@10"], 4),
                       "Recall@100": round(m["Recall@100"], 4),
                       "Tempo de busca (s)": round(t_busca, 1),
                       "Geração LLM (s)": None if t_geracao is None else round(t_geracao, 1)})
    rankings[metodo] = ranking
    print(f"{metodo:<22} nDCG@10={m['nDCG@10']:.4f}  Recall@100={m['Recall@100']:.4f}")


t0 = time.time()
bm25 = BM25Okapi([tokenizar(t) for t in doc_textos])
rk = bm25_rank(bm25, qids, queries, doc_ids)
registrar("BM25", rk, time.time() - t0)

encoders = {}
doc_embs = {}
for spec in ENCODERS:
    modelo = load_encoder(spec)
    encoders[spec["nome"]] = modelo
    t0 = time.time()
    doc_embs[spec["nome"]] = encode_texts(modelo, doc_textos, spec["d_prefix"], spec["normalize"])
    Q = encode_texts(modelo, [queries[q] for q in qids], spec["q_prefix"], spec["normalize"])
    rk = rank_from_vectors(Q, doc_embs[spec["nome"]], doc_ids, qids)
    registrar(f"Denso ({spec['nome']})", rk, time.time() - t0)

BM25                   nDCG@10=0.4255  Recall@100=0.7270


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (6 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Denso (mE5-small)      nDCG@10=0.5276  Recall@100=0.9028


## 8. Geração das passagens hipotéticas (dois prompts)

Cada prompt gera 4 passagens por consulta. O resultado fica salvo em JSON, e a geração não se repete se o arquivo existir.

In [ ]:
hips, t_ger = {}, {}
llm = None
for nome_p, prompt in PROMPTS.items():
    cache = f"hipoteses_quati_{nome_p}_{LLM_NOME.replace('/', '_')}_{N_HIPOTESES}_{len(qids)}.json"
    if os.path.exists(cache):
        with open(cache, encoding="utf-8") as f:
            hips[nome_p] = json.load(f)
        t_ger[nome_p] = None
        print(f"[{nome_p}] passagens carregadas do cache")
        continue
    if llm is None:
        llm = load_llm(LLM_NOME)
    t0 = time.time()
    saida = generate_hypotheses(llm, [prompt.format(queries[q]) for q in qids],
                                N_HIPOTESES, MAX_NEW_TOKENS, TEMPERATURA, seed=SEED)
    t_ger[nome_p] = time.time() - t0
    hips[nome_p] = {q: [h if h else queries[q] for h in hs] for q, hs in zip(qids, saida)}
    with open(cache, "w", encoding="utf-8") as f:
        json.dump(hips[nome_p], f, ensure_ascii=False)
    print(f"[{nome_p}] geração concluída em {t_ger[nome_p]:.0f} s")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (4 outputs left out)

Gerando passagens:   0%|          | 0/4 [00:00<?, ?it/s]

[PT] geração concluída em 50 s


Gerando passagens:   0%|          | 0/4 [00:00<?, ?it/s]

[EN-instr] geração concluída em 47 s


## 9. HyDE

In [ ]:
for nome_p in PROMPTS:
    for spec in ENCODERS:
        modelo = encoders[spec["nome"]]
        t0 = time.time()
        V = hyde_vectors(modelo, spec, qids, queries, hips[nome_p])
        rk = rank_from_vectors(V, doc_embs[spec["nome"]], doc_ids, qids)
        registrar(f"HyDE-{nome_p} ({spec['nome']})", rk, time.time() - t0, t_ger[nome_p])

HyDE-PT (mE5-small)    nDCG@10=0.5728  Recall@100=0.9240
HyDE-EN-instr (mE5-small) nDCG@10=0.5347  Recall@100=0.9310


## 10. Tabela de resultados

In [ ]:
tabela = pd.DataFrame(resultados)
tabela.to_csv("resultados_quati_subconjunto.csv", index=False)
tabela

,Método,nDCG@10,Recall@100,Tempo de busca (s),Geração LLM (s)
0,BM25,0.4255,0.7270,13.5,NaN
1,Denso (mE5-small),0.5276,0.9028,189.8,NaN
2,HyDE-PT (mE5-small),0.5728,0.9240,0.9,49.7
3,HyDE-EN-instr (mE5-small),0.5347,0.9310,0.9,46.9


## 11. A diferença é estatisticamente confiável?

Mesma análise da SciFact: nDCG@10 por consulta, diferença pareada, intervalo de confiança de 95% por bootstrap e p-valores. Com poucas consultas os intervalos ficam largos, então desconfie de diferenças pequenas.

In [ ]:
from scipy import stats

def ndcg_por_consulta(ranking):
    return np.array([ndcg_at_k(ranking[q], qrels[q], 10) for q in qids])

por_consulta = {nome: ndcg_por_consulta(rk) for nome, rk in rankings.items()}


def comparar(a, b, n_boot=10000, seed=SEED):
    d = por_consulta[a] - por_consulta[b]
    rng = np.random.default_rng(seed)
    medias = rng.choice(d, size=(n_boot, len(d)), replace=True).mean(axis=1)
    lo, hi = np.percentile(medias, [2.5, 97.5])
    p_t = stats.ttest_1samp(d, 0.0).pvalue if d.std() > 0 else 1.0
    try:
        p_w = stats.wilcoxon(d).pvalue
    except ValueError:
        p_w = 1.0
    return {"Comparação": f"{a} - {b}", "Diferença média": d.mean(),
            "IC 95% inferior": lo, "IC 95% superior": hi,
            "p (t pareado)": p_t, "p (Wilcoxon)": p_w,
            "Consultas melhor": int((d > 0).sum()), "Consultas pior": int((d < 0).sum()),
            "Empates": int((d == 0).sum())}


pares = []
for spec in ENCODERS:
    n = spec["nome"]
    for nome_p in PROMPTS:
        pares.append((f"HyDE-{nome_p} ({n})", f"Denso ({n})"))
        pares.append((f"HyDE-{nome_p} ({n})", "BM25"))
nomes_p = list(PROMPTS)
if len(nomes_p) == 2:
    n = ENCODERS[0]["nome"]
    pares.append((f"HyDE-{nomes_p[0]} ({n})", f"HyDE-{nomes_p[1]} ({n})"))

tabela_sig = pd.DataFrame([comparar(a, b) for a, b in pares])
tabela_sig.to_csv("significancia_quati_subconjunto.csv", index=False)
tabela_sig.round(4)

,Comparação,Diferença média,IC 95% inferior,IC 95% superior,p (t pareado),p (Wilcoxon),Consultas melhor,Consultas pior,Empates
0,HyDE-PT (mE5-small) - Denso (mE5-small),0.0452,0.0027,0.0841,0.0335,0.0123,33,15,1
1,HyDE-PT (mE5-small) - BM25,0.1473,0.0731,0.2233,0.0003,0.0005,35,13,1
2,HyDE-EN-instr (mE5-small) - Denso (mE5-small),0.0070,-0.0332,0.0420,0.7158,0.2592,26,22,1
3,HyDE-EN-instr (mE5-small) - BM25,0.1091,0.0340,0.1856,0.0066,0.0044,33,15,1
4,HyDE-PT (mE5-small) - HyDE-EN-instr (mE5-small),0.0382,0.0193,0.0579,0.0004,0.0012,30,17,2


## 12. Exemplo qualitativo (para os slides)

A consulta em que o melhor HyDE mais ganha da busca densa, e a em que ele menos ajuda.

In [ ]:
def trecho(d, n=160):
    t = corpus[d]["text"].replace("\n", " ")
    return t[:n] + ("..." if len(t) > n else "")


def mostrar(qid, hyde_nome, denso_nome, prompt_nome):
    print("Pergunta:", queries[qid])
    print("\nPassagem hipotética (1ª):\n", hips[prompt_nome][qid][0][:500])
    for nome in (denso_nome, hyde_nome):
        ranking = rankings[nome]
        top3 = [("[ok nota %d] " % qrels[qid][d] if d in qrels[qid] else "[x] ") + trecho(d) for d in ranking[qid][:3]]
        print(f"\n{nome} (nDCG@10 = {ndcg_at_k(ranking[qid], qrels[qid], 10):.2f}):", *top3, sep="\n  ")


enc = ENCODERS[0]["nome"]
melhor = max(PROMPTS, key=lambda p: tabela.set_index("Método").loc[f"HyDE-{p} ({enc})", "nDCG@10"])
h_nome, d_nome = f"HyDE-{melhor} ({enc})", f"Denso ({enc})"
ganho = por_consulta[h_nome] - por_consulta[d_nome]
i_ganha, i_perde = int(np.argmax(ganho)), int(np.argmin(ganho))

print("=" * 70, f"\nPROMPT {melhor}: CASO EM QUE O HyDE GANHA ({ganho[i_ganha]:+.2f})\n", "=" * 70, sep="")
mostrar(qids[i_ganha], h_nome, d_nome, melhor)
rotulo = "PIORA" if ganho[i_perde] < 0 else "MENOS AJUDA"
print("\n" + "=" * 70, f"\nCASO EM QUE O HyDE {rotulo} ({ganho[i_perde]:+.2f})\n", "=" * 70, sep="")
mostrar(qids[i_perde], h_nome, d_nome, melhor)

PROMPT PT: CASO EM QUE O HyDE GANHA (+0.30)
Pergunta: Por que os anos de 80 são considerados a década perdida no Brasil

Passagem hipotética (1ª):
 Os anos de 1980 foram significativamente marcados pelo desastre econômico brasileiro conhecido como "Crise do Petróleo" ou "Dinâmica Econômica", onde houve uma forte desaceleração na produção de petróleo e aumento da inflação nos Estados Unidos (1973-1985), levando a uma contração das economias emergentes, incluindo o Brasil.

Este período foi caracterizado por:
1. Desabastecimento de energia elétrica: Com as falhas nas operações da Eletrobrás, muitas residências brasileiras enfrentaram cortes e

Denso (mE5-small) (nDCG@10 = 0.34):
  [ok nota 1] "III - O crescimento da indústria do entretenimento, nos anos cinqüenta, através da expansão do rádio, da criação da televisão e da popularização do cinema. IV ...
  [ok nota 1] sociedade. Refiro-me ao legado da Era Vargas, ao seu modelo de desenvolvimento autárquico e ao seu Estado intervencionista

## 13. Guardar o subconjunto

Compacta o subconjunto para você baixar e reaproveitar (por exemplo, para a Pessoa B ou para o app). Baixe também os arquivos `resultados_quati_subconjunto.csv`, `significancia_quati_subconjunto.csv` e os `hipoteses_quati_*.json` antes de fechar o Colab.

In [ ]:
shutil.make_archive("quati_subconjunto_beir", "zip", "beir_data", "quati_sub")
print("Arquivo criado: quati_subconjunto_beir.zip")
try:
    from google.colab import files
    files.download("quati_subconjunto_beir.zip")
except Exception:
    pass

Arquivo criado: quati_subconjunto_beir.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notas para o relatório

- **Subconjunto:** o corpus tem as passagens julgadas mais uma amostra de distratores (`N_DISTRATORES`). Com menos distratores que o corpus de 1 milhão de passagens, os valores de nDCG@10 ficam mais altos. Não compare com o artigo do Quati. Compare BM25, denso e HyDE entre si.
- **Poucas consultas:** são cerca de 50 consultas, e os julgamentos de relevância vêm de um LLM (GPT-4), não de pessoas. Diga isso nas limitações.
- **Prompts:** os dois prompts são adaptações nossas. Relate qual funcionou melhor e que a diferença entre eles pode estar dentro do ruído (veja a última linha da tabela de significância).
- **BM25:** sem stemming e sem stopwords em português, então pode ficar abaixo de um BM25 bem configurado.
- **Modelos:** o mE5 é multilíngue. O Qwen2.5-1.5B escreve português, mas menos bem que inglês. Confira se as passagens geradas estão mesmo em português (veja a seção 12).
- **App em português:** `rag_core.Config(corpus_dir="beir_data/quati_sub", idioma="pt")`. Se o prompt `EN-instr` ganhar do `PT`, troque `PERFIS["pt"]["hyde"]` em `rag_core.py` por ele.